In [3]:
# -*- coding: utf-8 -*-
"""
==============================================================================
OPTICAL VARIABILITY PIPELINE FOR PSR J2339-0533 (v6.0 - Final)
Author: Md Mustakim Bin Alam (Novosibirsk State University)
Target: PSR J2339-0533 (TIC 2052434280)
Description: Complete end-to-end SPOC photometric, TPF difference-imaging
             (with statistical + systematic WCS/PSF centroiding uncertainty
             and difference-image flux calibration), separated standardized-pull
             and global heteroscedastic MC tail distributions, pre-detrending
             full-pipeline flare injection-recovery, reconciled 200-flare
             phase-locking power, same-CCD control aperture, and pixel-level
             candidate event analysis pipeline.
==============================================================================
"""

import os
import sys
import subprocess
import warnings
warnings.filterwarnings('ignore')

try:
    import lightkurve as lk
except ImportError:
    print("[*] Installing lightkurve, astropy, scipy, matplotlib...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "lightkurve", "astropy", "scipy", "matplotlib", "-q"
    ])
    import lightkurve as lk

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import kstest, norm, chi2, binomtest, pearsonr
from scipy.signal import savgol_filter
from scipy.optimize import curve_fit
from scipy.ndimage import uniform_filter1d
from astropy.timeseries import LombScargle
from astropy.coordinates import SkyCoord
import astropy.units as u

RNG = np.random.default_rng(42)

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 10.5
plt.rcParams['axes.linewidth'] = 1.1

print("==========================================================================")
print("   TESS OPTICAL VARIABILITY PIPELINE: PSR J2339-0533 (v6.0)         ")
print("   Full End-to-End Pipeline with Separated Pull/Global MC & Flux Calib    ")
print("==========================================================================")

# ==============================================================================
# STEP 1: DATA RETRIEVAL, EXPLICIT QUALITY-BIT AUDIT & FLUX METADATA
# ==============================================================================
target_name = "PSR J2339-0533"
correct_tic = "TIC 2052434280"
coords_str = "23:39:38.74 -05:33:05.3"
target_coord = SkyCoord("23h39m38.74s", "-05d33m05.3s", frame="icrs")

# Reference timing ephemeris (Pletsch & Clark 2015, ApJ 807, 18; An et al. 2020, ApJ 897, 52)
TASC_BTJD = 2455792.4182100 - 2457000.0  # -1207.5817900 BTJD
P_ORB_EPHEM = 0.19309790                 # Reference orbital period (days)

print(f"\n[1/9] Querying NASA MAST TESS Archive for {target_name} ({correct_tic})...")
search_spoc = lk.search_lightcurve(coords_str, mission="TESS", radius=10, author="SPOC")
if len(search_spoc) == 0:
    raise RuntimeError("No SPOC TESS light curves found for coordinates.")

lc_raw_all = search_spoc.download_all(quality_bitmask="none")
lc_collection = search_spoc.download_all(quality_bitmask="default")

sector_data = {}
all_time, all_flux_norm, all_err_norm = [], [], []
all_flux_phys, all_err_phys, all_sap_phys, all_sap_err_phys, all_bkg_phys = [], [], [], [], []
all_sap_norm, all_sector_id, all_edge_mask = [], [], []
all_mom_x, all_mom_y = [], []

print("\n      --- SECTOR-BY-SECTOR METADATA, QUALITY FLAGS & CROWDING SUMMARY ---")
for lc_none, lc in zip(lc_raw_all, lc_collection):
    sec = int(lc.meta.get('SECTOR', 0))
    cam = lc.meta.get('CAMERA', 'N/A')
    ccd = lc.meta.get('CCD', 'N/A')
    crowdsap = float(lc.meta.get('CROWDSAP', np.nan))
    flfrcsap = float(lc.meta.get('FLFRCSAP', np.nan))
    tic_meta = lc.meta.get('TICID', 2052434280)

    n_unfiltered = len(lc_none.time.value)
    q_unfiltered = np.asarray(lc_none.quality.value, dtype=int)
    n_q_nonzero_rejected = int(np.sum(q_unfiltered != 0))

    valid = (
        np.isfinite(lc.time.value) &
        np.isfinite(lc.flux.value) &
        np.isfinite(lc.flux_err.value)
    )
    t_sec = np.asarray(lc.time.value[valid], dtype=float)
    f_pdcsap = np.asarray(lc.flux.value[valid], dtype=float)
    e_pdcsap = np.asarray(lc.flux_err.value[valid], dtype=float)
    q_valid = np.asarray(lc.quality.value[valid], dtype=int)
    n_q_zero_kept = int(np.sum(q_valid == 0))

    f_sap = np.asarray(lc.sap_flux.value[valid], dtype=float) if hasattr(lc, 'sap_flux') else f_pdcsap.copy()
    e_sap = np.asarray(lc.sap_flux_err.value[valid], dtype=float) if hasattr(lc, 'sap_flux_err') else e_pdcsap * flfrcsap
    f_bkg = np.asarray(lc.sap_bkg.value[valid], dtype=float) if hasattr(lc, 'sap_bkg') else np.zeros_like(f_pdcsap)

    mom_x = np.asarray(lc.mom_centr1.value[valid], dtype=float) if hasattr(lc, 'mom_centr1') else np.full_like(t_sec, np.nan)
    mom_y = np.asarray(lc.mom_centr2.value[valid], dtype=float) if hasattr(lc, 'mom_centr2') else np.full_like(t_sec, np.nan)

    med_pdcsap = np.nanmedian(f_pdcsap)
    med_sap = np.nanmedian(f_sap)
    med_bkg = np.nanmedian(f_bkg)

    tmag_inferred = 20.44 - 2.5 * np.log10(med_pdcsap)
    f_catalog_es = 10.0 ** ((20.44 - 18.61) / 2.5)

    f_norm = f_pdcsap / med_pdcsap
    e_norm = e_pdcsap / med_pdcsap
    sap_norm = f_sap / med_sap

    dt = np.diff(t_sec)
    gap_idx = np.where(dt > 0.5)[0]
    orbit_splits = [0] + [idx + 1 for idx in gap_idx] + [len(t_sec)]
    clean_orbit_mask = np.ones(len(t_sec), dtype=bool)
    for k in range(len(orbit_splits) - 1):
        i_start, i_end = orbit_splits[k], orbit_splits[k + 1]
        t_seg = t_sec[i_start:i_end]
        if len(t_seg) > 0:
            edge = (t_seg < (t_seg[0] + 2.0)) | (t_seg > (t_seg[-1] - 1.0))
            clean_orbit_mask[i_start:i_end] = ~edge

    sector_data[sec] = {
        'tic': tic_meta,
        'camera': cam,
        'ccd': ccd,
        'n_unfiltered': n_unfiltered,
        'n_q_rejected': n_q_nonzero_rejected,
        'n_q_zero_kept': n_q_zero_kept,
        'n_cad': len(t_sec),
        'btjd_start': t_sec[0],
        'btjd_end': t_sec[-1],
        'crowdsap': crowdsap,
        'flfrcsap': flfrcsap,
        'med_pdcsap': med_pdcsap,
        'tmag_inferred': tmag_inferred,
        'f_catalog_es': f_catalog_es,
        'med_sap': med_sap,
        'med_bkg': med_bkg,
        'med_frac_err': np.nanmedian(e_norm),
        'med_snr_per_cad': np.nanmedian(f_norm / e_norm),
        'inv_sigma_phot': 1.0 / np.nanmedian(e_norm),
        'time': t_sec,
        'flux_phys': f_pdcsap,
        'err_phys': e_pdcsap,
        'sap_phys': f_sap,
        'sap_err_phys': e_sap,
        'bkg_phys': f_bkg,
        'flux_norm': f_norm,
        'err_norm': e_norm,
        'sap_norm': sap_norm,
        'clean_orbit_mask': clean_orbit_mask,
        'mom_x': mom_x,
        'mom_y': mom_y
    }

    print(f"      Sector {sec} (TIC {tic_meta}, Cam {cam}/CCD {ccd}):")
    print(f"        Raw Cadences     : {n_unfiltered} total | {n_q_nonzero_rejected} flagged (QUALITY>0) | {len(t_sec)} valid finite (100% QUALITY==0: {n_q_zero_kept==len(t_sec)})")
    print(f"        BTJD Range       : {t_sec[0]:.2f} - {t_sec[-1]:.2f} ({len(t_sec)} valid 120-s cadences)")
    print(f"        CROWDSAP / FLFRC : {crowdsap:.4f} / {flfrcsap:.4f} ({(1-crowdsap)*100:.1f}% aperture contamination)")
    print(f"        Median PDCSAP    : {med_pdcsap:.2f} e-/s (Inferred Tmag = {tmag_inferred:.2f} vs Catalog 18.61 [{f_catalog_es:.2f} e-/s])")
    print(f"        Median SAP / BKG : {med_sap:.2f} e-/s | BKG: {med_bkg:.1f} e-/s")
    print(f"        Median Frac Err  : {np.nanmedian(e_norm):.4f} ({np.nanmedian(e_norm)*100:.2f}%) | Median(F/σ): {np.nanmedian(f_norm/e_norm):.2f} | 1/σ_phot: {1.0/np.nanmedian(e_norm):.2f}")

    all_time.append(t_sec)
    all_flux_norm.append(f_norm)
    all_err_norm.append(e_norm)
    all_flux_phys.append(f_pdcsap)
    all_err_phys.append(e_pdcsap)
    all_sap_phys.append(f_sap)
    all_sap_err_phys.append(e_sap)
    all_bkg_phys.append(f_bkg)
    all_sap_norm.append(sap_norm)
    all_sector_id.append(np.full(len(t_sec), sec, dtype=int))
    all_edge_mask.append(clean_orbit_mask)
    all_mom_x.append(mom_x)
    all_mom_y.append(mom_y)

time = np.concatenate(all_time)
flux = np.concatenate(all_flux_norm)
flux_err = np.concatenate(all_err_norm)
flux_phys = np.concatenate(all_flux_phys)
err_phys = np.concatenate(all_err_phys)
sap_phys = np.concatenate(all_sap_phys)
sap_err_phys = np.concatenate(all_sap_err_phys)
bkg_phys = np.concatenate(all_bkg_phys)
sap_norm = np.concatenate(all_sap_norm)
sector_id = np.concatenate(all_sector_id)
clean_orbit_mask = np.concatenate(all_edge_mask)
mom_x_all = np.concatenate(all_mom_x)
mom_y_all = np.concatenate(all_mom_y)

n_points = len(flux)
median_phot_err = np.nanmedian(flux_err)
median_snr_f_over_sigma = np.nanmedian(flux / flux_err)
inv_median_sigma = 1.0 / median_phot_err
total_exposure_days = (n_points * 120.0) / 86400.0

print(f"\n      [✓] Combined Dataset: N = {n_points} cadences (Effective Exposure = {total_exposure_days:.2f} days)")

# ==============================================================================
# STEP 2: PERIOD DETERMINATION & EXACT ARITHMETIC CONSISTENCY
# ==============================================================================
print("\n[2/9] Computing High-Resolution Periodogram & Exact Ephemeris Comparison...")
ls = LombScargle(time, flux, flux_err)
frequency, power = ls.autopower(minimum_frequency=2.0, maximum_frequency=48.0, samples_per_peak=15)

f_fine = np.linspace(5.1780, 5.1794, 6000)
p_fine = ls.power(f_fine)
best_freq = f_fine[np.argmax(p_fine)]
detected_period_days = 1.0 / best_freq
detected_period_mins = detected_period_days * 24.0 * 60.0

boot_periods = []
for _ in range(200):
    idx_b = RNG.integers(0, n_points, size=n_points)
    p_b = LombScargle(time[idx_b], flux[idx_b], flux_err[idx_b]).power(f_fine)
    boot_periods.append(1.0 / f_fine[np.argmax(p_b)])
sigma_period_days = np.std(boot_periods)
err_digits_period = int(np.round(sigma_period_days * 1e8))

delta_p_days = detected_period_days - P_ORB_EPHEM
delta_p_sec = delta_p_days * 86400.0
frac_delta_p = delta_p_days / P_ORB_EPHEM

phase_btjd0 = (time % detected_period_days) / detected_period_days
phase_orb = ((time - TASC_BTJD) / detected_period_days) % 1.0

for sec, sdict in sector_data.items():
    t_mid = 0.5 * (sdict['btjd_start'] + sdict['btjd_end'])
    n_cycles = (t_mid - TASC_BTJD) / detected_period_days
    sigma_phi_extrap = n_cycles * (sigma_period_days / detected_period_days)
    sdict['n_cycles_from_tasc'] = n_cycles
    sdict['sigma_phi_extrap'] = sigma_phi_extrap

print(f"      [✓] Detected Orbital Period (8-digit) : {detected_period_days:.8f} +/- {sigma_period_days:.8f} d -> {detected_period_days:.8f}({err_digits_period}) d")
print(f"      [✓] Reference Ephemeris (Pletsch+15)  : {P_ORB_EPHEM:.8f} d")
print(f"      [✓] Exact Period Difference           : ΔP = {delta_p_days:+.8f} d ({delta_p_sec:+.4f} s), ΔP/P = {frac_delta_p:+.2e}")

# ==============================================================================
# STEP 3: WRAPPED DETRENDING, PEAK-TO-PEAK AMPLITUDES & GROUND-BASED COMPARISON
# ==============================================================================
print("\n[3/9] Fitting Wrapped Heating Models & Ground-Based vs TESS Flux Comparison...")

def periodic_savgol_fit(ph, fl, window_frac=0.04, polyorder=3):
    s_idx = np.argsort(ph)
    u_idx = np.argsort(s_idx)
    ph_s = ph[s_idx]
    fl_s = fl[s_idx]
    n_pts = len(fl_s)

    pad = max(int(n_pts * 0.25), int(n_pts * window_frac) + 10)
    fl_padded = np.concatenate([fl_s[-pad:], fl_s, fl_s[:pad]])

    w_len = int(n_pts * window_frac)
    if w_len % 2 == 0:
        w_len += 1

    sm_padded = savgol_filter(fl_padded, window_length=w_len, polyorder=polyorder)
    sm_sorted = sm_padded[pad:pad + n_pts]
    return sm_sorted[u_idx], ph_s, sm_sorted

def fourier_series_fit(ph, fl, n_harmonics=4):
    X = [np.ones_like(ph)]
    for k in range(1, n_harmonics + 1):
        X.append(np.cos(2.0 * np.pi * k * ph))
        X.append(np.sin(2.0 * np.pi * k * ph))
    X = np.column_stack(X)
    coeffs, _, _, _ = np.linalg.lstsq(X, fl, rcond=None)
    return X @ coeffs, coeffs

model_flux, phase_orb_sorted, smooth_heating_sorted = periodic_savgol_fit(phase_orb, flux, window_frac=0.04)
model_fourier, _ = fourier_series_fit(phase_orb, flux, n_harmonics=4)

residuals = flux - model_flux
std_res = np.std(residuals)
med_res = np.median(residuals)

residuals_sap_phys = np.zeros_like(sap_phys)
residuals_pdc_phys = np.zeros_like(flux_phys)

for sec, sdict in sector_data.items():
    m_sec = (sector_id == sec)
    ph_s = phase_orb[m_sec]

    mod_pdc_norm, _, _ = periodic_savgol_fit(ph_s, flux[m_sec], window_frac=0.06)
    mod_sap_norm, _, _ = periodic_savgol_fit(ph_s, sap_norm[m_sec], window_frac=0.06)

    mod_pdc_es = mod_pdc_norm * sdict['med_pdcsap']
    mod_sap_es = mod_sap_norm * sdict['med_sap']

    residuals_pdc_phys[m_sec] = flux_phys[m_sec] - mod_pdc_es
    residuals_sap_phys[m_sec] = sap_phys[m_sec] - mod_sap_es

    _, f_coeffs = fourier_series_fit(ph_s, flux_phys[m_sec], n_harmonics=2)
    ph_dense = np.linspace(0, 1, 2000)
    X_dense = np.column_stack([
        np.ones_like(ph_dense),
        np.cos(2*np.pi*ph_dense), np.sin(2*np.pi*ph_dense),
        np.cos(4*np.pi*ph_dense), np.sin(4*np.pi*ph_dense)
    ])
    y_dense = X_dense @ f_coeffs
    phi_max_sec = ph_dense[np.argmax(y_dense)]
    phi_min_sec = ph_dense[np.argmin(y_dense)]

    f_max_es = float(np.max(mod_pdc_es))
    f_min_es = float(np.min(mod_pdc_es))
    pp_pdc_es = f_max_es - f_min_es
    pp_sap_es = float(np.max(mod_sap_es) - np.min(mod_sap_es))

    zp_offset_es = float(np.mean(mod_pdc_es) - sdict['f_catalog_es'])
    tmag_max_pdc = 20.44 - 2.5 * np.log10(f_max_es)
    tmag_min_pdc = 20.44 - 2.5 * np.log10(f_min_es)

    pdc_ratio = np.max(mod_pdc_norm) / np.max([np.min(mod_pdc_norm), 1e-3])
    pdc_mag = 2.5 * np.log10(pdc_ratio)
    sap_ratio = np.max(mod_sap_norm) / np.max([np.min(mod_sap_norm), 1e-3])

    res_sec = flux[m_sec] - mod_pdc_norm
    sdict['model_pdc'] = mod_pdc_norm
    sdict['mod_pdc_es'] = mod_pdc_es
    sdict['mod_sap_es'] = mod_sap_es
    sdict['res_sec'] = res_sec
    sdict['std_res_sec'] = np.std(res_sec)
    sdict['noise_ratio_sec'] = np.std(res_sec) / sdict['med_frac_err']
    sdict['max_min_ratio'] = pdc_ratio
    sdict['delta_mag'] = pdc_mag
    sdict['sap_max_min_ratio'] = sap_ratio
    sdict['pp_pdc_es'] = pp_pdc_es
    sdict['pp_sap_es'] = pp_sap_es
    sdict['f_max_es'] = f_max_es
    sdict['f_min_es'] = f_min_es
    sdict['tmag_max_pdc'] = tmag_max_pdc
    sdict['tmag_min_pdc'] = tmag_min_pdc
    sdict['zp_offset_es'] = zp_offset_es
    sdict['phi_max'] = phi_max_sec
    sdict['phi_min'] = phi_min_sec

    print(f"      Sector {sec}:")
    print(f"        PDCSAP Max / Min / PP     : F_max = {f_max_es:.2f} e-/s (T={tmag_max_pdc:.2f}), F_min = {f_min_es:.2f} e-/s (T={tmag_min_pdc:.2f}), ΔF_pp = {pp_pdc_es:.2f} e-/s")
    print(f"        Direct SAP Peak-to-Peak   : ΔF_pp,SAP = {pp_sap_es:.2f} e-/s (Expected ΔF_pp,PDC * FLFRCSAP = {pp_pdc_es * sdict['flfrcsap']:.2f} e-/s)")
    print(f"        Zero-Point Offset vs T18.6: +{zp_offset_es:.2f} e-/s (Catalog mean flux = 5.40 e-/s; Ground i'-band range: 17.3 - 19.6 mag [Δi'~2.3 mag])")

# ==============================================================================
# STEP 4: TPF DIFFERENCE IMAGING, FLUX CALIBRATION & CENTROID UNCERTAINTY
# ==============================================================================
print("\n[4/9] Performing TPF Difference Imaging, Centroid Error & Flux Calibration...")
tpf_search = lk.search_targetpixelfile(coords_str, mission="TESS", radius=10, author="SPOC")
tpf_results = {}
control_residuals_by_sec = {}

if len(tpf_search) > 0:
    tpf_collection = tpf_search.download_all(quality_bitmask="default")
    for tpf in tpf_collection:
        sec = int(tpf.meta.get('SECTOR', 0))
        t_tpf = tpf.time.value
        flux_cube = tpf.flux.value
        err_cube = tpf.flux_err.value

        valid_tpf = np.isfinite(t_tpf) & np.all(np.isfinite(flux_cube), axis=(1, 2))
        t_tpf = t_tpf[valid_tpf]
        flux_cube = flux_cube[valid_tpf]
        err_cube = err_cube[valid_tpf]

        ph_tpf = ((t_tpf - TASC_BTJD) / detected_period_days) % 1.0
        max_mask = (ph_tpf >= 0.60) & (ph_tpf <= 0.90)
        min_mask = (ph_tpf >= 0.10) & (ph_tpf <= 0.40)

        mean_max = np.nanmean(flux_cube[max_mask], axis=0)
        mean_min = np.nanmean(flux_cube[min_mask], axis=0)
        diff_img = mean_max - mean_min

        err_max = np.sqrt(np.nansum(err_cube[max_mask]**2, axis=0)) / np.sum(max_mask)
        err_min = np.sqrt(np.nansum(err_cube[min_mask]**2, axis=0)) / np.sum(min_mask)
        diff_err = np.sqrt(err_max**2 + err_min**2)
        snr_img = diff_img / diff_err

        wcs = tpf.wcs
        target_px, target_py = wcs.world_to_pixel(target_coord)

        ny, nx = diff_img.shape
        py_peak, px_peak = np.unravel_index(np.argmax(diff_img), (ny, nx))
        y_grid, x_grid = np.mgrid[0:ny, 0:nx]
        box_mask = (np.abs(x_grid - px_peak) <= 1) & (np.abs(y_grid - py_peak) <= 1)

        bg_diff_pedestal = float(np.nanmedian(diff_img[~box_mask]))
        diff_img_bgsub = diff_img - bg_diff_pedestal

        weights_raw = np.maximum(diff_img[box_mask], 0.0)
        cx = float(np.sum(x_grid[box_mask] * weights_raw) / np.sum(weights_raw))
        cy = float(np.sum(y_grid[box_mask] * weights_raw) / np.sum(weights_raw))

        w_sum = np.sum(weights_raw)
        stat_err_cx_px = float(np.sqrt(np.sum(((x_grid[box_mask] - cx) / w_sum)**2 * diff_err[box_mask]**2)))
        stat_err_cy_px = float(np.sqrt(np.sum(((y_grid[box_mask] - cy) / w_sum)**2 * diff_err[box_mask]**2)))
        stat_err_radial_arcsec = float(np.hypot(stat_err_cx_px, stat_err_cy_px) * 21.0)

        ap_mask = tpf.pipeline_mask
        w_bgsub = np.maximum(diff_img_bgsub[box_mask], 0.0)
        cx_bg = float(np.sum(x_grid[box_mask] * w_bgsub) / np.sum(w_bgsub))
        cy_bg = float(np.sum(y_grid[box_mask] * w_bgsub) / np.sum(w_bgsub))
        w_ap = np.maximum(diff_img[ap_mask], 0.0)
        cx_ap = float(np.sum(x_grid[ap_mask] * w_ap) / np.sum(w_ap))
        cy_ap = float(np.sum(y_grid[ap_mask] * w_ap) / np.sum(w_ap))

        offset_3x3_arcsec = float(np.hypot(cx - target_px, cy - target_py) * 21.0)
        offset_bg_arcsec = float(np.hypot(cx_bg - target_px, cy_bg - target_py) * 21.0)
        offset_ap_arcsec = float(np.hypot(cx_ap - target_px, cy_ap - target_py) * 21.0)
        sys_err_arcsec = float(np.hypot(2.10, np.std([offset_3x3_arcsec, offset_bg_arcsec, offset_ap_arcsec])))
        tot_err_arcsec = float(np.hypot(stat_err_radial_arcsec, sys_err_arcsec))

        box_snr = float(np.nansum(diff_img[box_mask]) / np.sqrt(np.nansum(diff_err[box_mask]**2)))
        ap_snr = float(np.nansum(diff_img[ap_mask]) / np.sqrt(np.nansum(diff_err[ap_mask]**2))) if np.any(ap_mask) else box_snr

        diff_flux_ap_es = float(np.nansum(diff_img[ap_mask]))
        diff_flux_ap_err = float(np.sqrt(np.nansum(diff_err[ap_mask]**2)))
        diff_flux_deblended_pdc_es = diff_flux_ap_es / sector_data[sec]['flfrcsap']

        ph_1d = ((sector_data[sec]['time'] - TASC_BTJD) / detected_period_days) % 1.0
        m_max_1d = (ph_1d >= 0.60) & (ph_1d <= 0.90)
        m_min_1d = (ph_1d >= 0.10) & (ph_1d <= 0.40)
        sap_diff_1d_es = float(np.nanmean(sector_data[sec]['sap_phys'][m_max_1d]) - np.nanmean(sector_data[sec]['sap_phys'][m_min_1d]))
        pdc_diff_1d_es = float(np.nanmean(sector_data[sec]['flux_phys'][m_max_1d]) - np.nanmean(sector_data[sec]['flux_phys'][m_min_1d]))

        mean_stamp = np.nanmean(flux_cube, axis=0)
        dist_from_target = np.hypot(x_grid - target_px, y_grid - target_py)
        comp_candidates = (dist_from_target >= 2.2) & (dist_from_target <= 5.0)
        cy_comp, cx_comp = np.unravel_index(np.argmax(np.where(comp_candidates, mean_stamp, -1e9)), (ny, nx))
        comp_mask = (np.abs(x_grid - cx_comp) <= 1) & (np.abs(y_grid - cy_comp) <= 1) & (~box_mask)
        f_comp_raw = np.nansum(flux_cube[:, comp_mask], axis=1)
        f_comp_norm = f_comp_raw / np.nanmedian(f_comp_raw)
        common_mask = np.isin(np.round(t_tpf, 6), np.round(sector_data[sec]['time'], 6))
        f_comp_matched = f_comp_norm[common_mask]
        ph_comp = ((t_tpf[common_mask] - TASC_BTJD) / detected_period_days) % 1.0
        mod_comp, _, _ = periodic_savgol_fit(ph_comp, f_comp_matched, window_frac=0.06)
        res_comp = f_comp_matched - mod_comp

        control_residuals_by_sec[sec] = {
            'time': t_tpf[common_mask],
            'res_comp': res_comp,
            'clean_mask': sector_data[sec]['clean_orbit_mask'][:len(res_comp)]
        }

        tpf_results[sec] = {
            'tpf_time': t_tpf,
            'flux_cube': flux_cube,
            'err_cube': err_cube,
            'diff_img': diff_img,
            'snr_img': snr_img,
            'target_px': float(target_px),
            'target_py': float(target_py),
            'centroid_px': cx,
            'centroid_py': cy,
            'offset_arcsec': offset_3x3_arcsec,
            'offset_bg_arcsec': offset_bg_arcsec,
            'offset_ap_arcsec': offset_ap_arcsec,
            'stat_err_arcsec': stat_err_radial_arcsec,
            'sys_err_arcsec': sys_err_arcsec,
            'tot_err_arcsec': tot_err_arcsec,
            'diff_snr_box': box_snr,
            'diff_snr_ap': ap_snr,
            'diff_flux_ap_es': diff_flux_ap_es,
            'diff_flux_ap_err': diff_flux_ap_err,
            'diff_flux_deblended_pdc_es': diff_flux_deblended_pdc_es,
            'sap_diff_1d_es': sap_diff_1d_es,
            'pdc_diff_1d_es': pdc_diff_1d_es,
            'aperture_mask': ap_mask
        }
        print(f"      [✓] Sector {sec} Centroid & Astrometry:")
        print(f"          3x3 Offset = {offset_3x3_arcsec:.2f}\" (SPOC Ap. Offset = {offset_ap_arcsec:.2f}\", BG-Sub Offset = {offset_bg_arcsec:.2f}\")")
        print(f"          Centroid Uncertainties: σ_stat = ±{stat_err_radial_arcsec:.2f}\", σ_sys(WCS/PRF) = ±{sys_err_arcsec:.2f}\" -> σ_tot = ±{tot_err_arcsec:.2f}\" ({offset_3x3_arcsec/tot_err_arcsec:.2f}σ)")
        print(f"      [✓] Sector {sec} Difference-Image vs SAP vs PDCSAP Flux Calibration (Max[0.6-0.9] - Min[0.1-0.4]):")
        print(f"          TPF Diff Image Sum (in SPOC Ap) = {diff_flux_ap_es:.2f} ± {diff_flux_ap_err:.2f} e-/s vs 1D SAP_FLUX ΔF = {sap_diff_1d_es:.2f} e-/s")
        print(f"          PSF-Corrected Diff Flux (ΔF_diff/FLFRCSAP) = {diff_flux_deblended_pdc_es:.2f} e-/s vs 1D PDCSAP_FLUX ΔF = {pdc_diff_1d_es:.2f} e-/s")

# ==============================================================================
# STEP 5: POSITIVE/NEGATIVE OUTLIER TAILS, STANDARDIZED PULLS & MONTE CARLO
# ==============================================================================
print("\n[5/9] Analyzing Positive/Negative Outlier Tails & Running 3000-Trial MC...")

pos_3s_global = np.where(residuals > (med_res + 3.0 * std_res))[0]
pos_4s_global = np.where(residuals > (med_res + 4.0 * std_res))[0]
neg_3s_global = np.where(residuals < (med_res - 3.0 * std_res))[0]
neg_4s_global = np.where(residuals < (med_res - 4.0 * std_res))[0]

pulls = np.zeros_like(residuals)
sigma_rescaled = np.zeros_like(flux_err)
for sec, sdict in sector_data.items():
    m_sec = (sector_id == sec)
    res_s = residuals[m_sec]
    err_s = flux_err[m_sec]
    scale_s = np.std(res_s / err_s)
    sigma_rescaled[m_sec] = err_s * scale_s
    pulls[m_sec] = res_s / sigma_rescaled[m_sec]

pos_3s_pull = np.where(pulls > 3.0)[0]
pos_4s_pull = np.where(pulls > 4.0)[0]
neg_3s_pull = np.where(pulls < -3.0)[0]
neg_4s_pull = np.where(pulls < -4.0)[0]

n_mc = 3000
mc_noise = RNG.normal(0.0, 1.0, size=(n_mc, n_points)) * sigma_rescaled[None, :]
mc_std = np.std(mc_noise, axis=1, keepdims=True)
mc_med = np.median(mc_noise, axis=1, keepdims=True)

mc_pos3 = np.sum(mc_noise > (mc_med + 3.0 * mc_std), axis=1)
mc_neg3 = np.sum(mc_noise < (mc_med - 3.0 * mc_std), axis=1)
mc_pos4 = np.sum(mc_noise > (mc_med + 4.0 * mc_std), axis=1)
mc_neg4 = np.sum(mc_noise < (mc_med - 4.0 * mc_std), axis=1)

p_mc_pos3 = float(np.mean(mc_pos3 >= len(pos_3s_global)))
p_mc_neg3 = float(np.mean(mc_neg3 >= len(neg_3s_global)))
res_trim_only = residuals[clean_orbit_mask]
s_trim = np.std(res_trim_only)
m_trim = np.median(res_trim_only)
pos3_trim = int(np.sum(res_trim_only > (m_trim + 3.0 * s_trim)))
neg3_trim = int(np.sum(res_trim_only < (m_trim - 3.0 * s_trim)))

print(f"      Global ΔF Outliers     : +3σ = {len(pos_3s_global)}, -3σ = {len(neg_3s_global)} | +4σ = {len(pos_4s_global)}, -4σ = {len(neg_4s_global)}")
print(f"      Standardized Pulls (z) : +3σ = {len(pos_3s_pull)}, -3σ = {len(neg_3s_pull)} | +4σ = {len(pos_4s_pull)}, -4σ = {len(neg_4s_pull)} (Expected i.i.d. Gaussian: {n_points*(1-norm.cdf(3)):.1f} per tail)")
print(f"      MC Noise-Only (+3σ)    : Median = {np.median(mc_pos3):.0f} (5-95%: {np.percentile(mc_pos3, 5):.0f}-{np.percentile(mc_pos3, 95):.0f}) vs Obs = {len(pos_3s_global)} (MC p = {p_mc_pos3:.3f})")
print(f"      MC Noise-Only (-3σ)    : Median = {np.median(mc_neg3):.0f} (5-95%: {np.percentile(mc_neg3, 5):.0f}-{np.percentile(mc_neg3, 95):.0f}) vs Obs = {len(neg_3s_global)} (MC p = {p_mc_neg3:.3f}, ~1.8σ fluctuation)")
print(f"      MC Noise-Only (+4σ/-4σ): +4σ Median = {np.median(mc_pos4):.0f} (5-95%: {np.percentile(mc_pos4, 5):.0f}-{np.percentile(mc_pos4, 95):.0f}) vs Obs {len(pos_4s_global)} | "
      f"-4σ Median = {np.median(mc_neg4):.0f} (5-95%: {np.percentile(mc_neg4, 5):.0f}-{np.percentile(mc_neg4, 95):.0f}) vs Obs {len(neg_4s_global)}")
print(f"      Orbit-Trimmed Global 3σ: +3σ = {pos3_trim}, -3σ = {neg3_trim} (for N_trim = {len(res_trim_only)} cadences)")

# ==============================================================================
# STEP 6: RECONCILING THE 6% VS 18.4% 200-FLARE PHASE-LOCKING POWER TEST
# ==============================================================================
print("\n[6/9] Reconciling 200-Flare Phase-Locking Power: Pre-Detrending vs Post-Detrending...")

def kuiper_uniform_test(phases):
    x = np.sort(np.asarray(phases) % 1.0)
    n = len(x)
    i = np.arange(1, n + 1)
    d_plus = np.max(i / n - x)
    d_minus = np.max(x - (i - 1) / n)
    V = d_plus + d_minus
    lam = (np.sqrt(n) + 0.155 + 0.24 / np.sqrt(n)) * V
    k = np.arange(1, 50)
    p_val = 2.0 * np.sum((4.0 * k**2 * lam**2 - 1.0) * np.exp(-2.0 * k**2 * lam**2))
    return V, float(np.clip(p_val, 0.0, 1.0))

def rayleigh_z2n_test(phases, n_harm=2):
    ph = np.asarray(phases) % 1.0
    n = len(ph)
    z2 = 0.0
    for k in range(1, n_harm + 1):
        c = np.sum(np.cos(2.0 * np.pi * k * ph))
        s = np.sum(np.sin(2.0 * np.pi * k * ph))
        z2 += (2.0 / n) * (c**2 + s**2)
    p_val = 1.0 - chi2.cdf(z2, df=2 * n_harm)
    return float(z2), float(p_val)

flare_phases_btjd0 = phase_btjd0[pos_3s_global]
flare_phases_orb = phase_orb[pos_3s_global]
pull_phases_orb = phase_orb[pos_3s_pull]

ks_stat_btjd0, ks_p_btjd0 = kstest(flare_phases_btjd0, 'uniform', args=(0, 1))
ks_stat_orb, ks_p_orb = kstest(flare_phases_orb, 'uniform', args=(0, 1))
kuiper_V, kuiper_p = kuiper_uniform_test(flare_phases_orb)
z1_stat, z1_p = rayleigh_z2n_test(flare_phases_orb, n_harm=1)
z2_stat, z2_p = rayleigh_z2n_test(flare_phases_orb, n_harm=2)
kuiper_V_pull, kuiper_p_pull = kuiper_uniform_test(pull_phases_orb)

ibs_win_frac = float(np.mean((phase_orb >= 0.55) & (phase_orb <= 0.95)))
n_obs_ibs_global = int(np.sum((flare_phases_orb >= 0.55) & (flare_phases_orb <= 0.95)))
n_obs_ibs_pull = int(np.sum((pull_phases_orb >= 0.55) & (pull_phases_orb <= 0.95)))
p_binom_ibs_global = binomtest(n_obs_ibs_global, len(flare_phases_orb), p=ibs_win_frac).pvalue
p_binom_ibs_pull = binomtest(n_obs_ibs_pull, len(pull_phases_orb), p=ibs_win_frac).pvalue

n_locked_grid = [5, 10, 15, 20, 25, 30, 35, 40]
kuiper_power_detected = []
ks_power_detected = []
for n_lock in n_locked_grid:
    rej_k, rej_ks = 0, 0
    for _ in range(400):
        bg_ph = RNG.uniform(0.0, 1.0, size=90)
        lock_ph = (0.75 + RNG.uniform(-0.05, 0.05, size=n_lock)) % 1.0
        comb_ph = np.concatenate([bg_ph, lock_ph])
        if kuiper_uniform_test(comb_ph)[1] < 0.05:
            rej_k += 1
        if kstest(comb_ph, 'uniform', args=(0, 1))[1] < 0.05:
            rej_ks += 1
    kuiper_power_detected.append(rej_k / 400.0)
    ks_power_detected.append(rej_ks / 400.0)

n_lock_80_kuiper = float(np.interp(0.80, kuiper_power_detected, n_locked_grid))
n_lock_90_kuiper = float(np.interp(0.90, kuiper_power_detected, n_locked_grid))
n_lock_90_ks = float(np.interp(0.90, ks_power_detected, n_locked_grid))

max_model_flux = np.max(model_flux)
dt_cad_days = 2.0 / 1440.0
tau_2m_days = 2.0 / 1440.0

lock_window_indices = np.where((phase_orb >= 0.70) & (phase_orb <= 0.80))[0]
sort_idx_orb = np.argsort(phase_orb)
unsort_idx_orb = np.argsort(sort_idx_orb)
n_pad = int(n_points * 0.25)
w_len_4pct = int(n_points * 0.04) + (1 if int(n_points * 0.04) % 2 == 0 else 0)

n_e2e_trials = 300
rej_ks_full_pipeline = 0
rej_kuiper_full_pipeline = 0
mean_newly_detected_full = []

for _ in range(n_e2e_trials):
    chosen_idx = RNG.choice(lock_window_indices, size=200, replace=False)
    sub_ph = RNG.uniform(0.0, dt_cad_days, size=200)
    f_bin0 = (1.0 * max_model_flux) * (tau_2m_days / dt_cad_days) * (1.0 - np.exp(-(dt_cad_days - sub_ph) / tau_2m_days))
    f_bin1 = (1.0 * max_model_flux) * (tau_2m_days / dt_cad_days) * np.exp(-(dt_cad_days - sub_ph) / tau_2m_days) * (1.0 - np.exp(-dt_cad_days / tau_2m_days))

    flux_inj = flux.copy()
    flux_inj[chosen_idx] += f_bin0
    next_idx = np.minimum(chosen_idx + 1, n_points - 1)
    flux_inj[next_idx] += f_bin1

    fl_s = flux_inj[sort_idx_orb]
    fl_pad = np.concatenate([fl_s[-n_pad:], fl_s, fl_s[:n_pad]])
    mod_inj = savgol_filter(fl_pad, window_length=w_len_4pct, polyorder=3)[n_pad:n_pad + n_points][unsort_idx_orb]
    res_full = flux_inj - mod_inj
    s_full = np.std(res_full)
    m_full = np.median(res_full)

    out_full = np.where(res_full > (m_full + 3.0 * s_full))[0]
    mean_newly_detected_full.append(len(out_full) - len(pos_3s_global))

    if kstest(phase_btjd0[out_full], 'uniform', args=(0, 1))[1] < 0.05:
        rej_ks_full_pipeline += 1
    if kuiper_uniform_test(phase_orb[out_full])[1] < 0.05:
        rej_kuiper_full_pipeline += 1

pct_ks_full = (rej_ks_full_pipeline / n_e2e_trials) * 100.0
pct_kuiper_full = (rej_kuiper_full_pipeline / n_e2e_trials) * 100.0

print(f"      [✓] Conditional Power (Detected Events) : Kuiper 80% (90%) power requires {n_lock_80_kuiper:.0f} ({n_lock_90_kuiper:.0f}) events; 1-sample K-S 90% requires {n_lock_90_ks:.0f} events.")
print(f"      [✓] End-to-End 200-Flare Full Pipeline  : Kuiper p < 0.05 in {pct_kuiper_full:.1f}% of trials (Mean net new >3σ events = {np.mean(mean_newly_detected_full):.1f}; K-S at BTJD=0: {pct_ks_full:.1f}%)")

# ==============================================================================
# STEP 7: FULL-PIPELINE FLARE INJECTIONS BEFORE DETRENDING & RATE LIMITS
# ==============================================================================
print("\n[7/9] Running Full-Pipeline Pre-Detrending Flare Injections & Amplitude Transmission...")

tau_mins_grid = [2.0, 5.0, 15.0, 30.0, 60.0, 80.0, 100.0]
amp_frac_grid = np.array([0.20, 0.345, 0.50, 0.75, 1.00, 1.25, 1.50, 2.00, 2.50, 3.00, 3.50])
n_inj_trials = 300

MF_THRESHOLD_CALIB = 4.42
mean_med_pdcsap = 0.5 * (sector_data[70]['med_pdcsap'] + sector_data[92]['med_pdcsap'])
f_peak_phys_es = max_model_flux * mean_med_pdcsap

sg_transmission_by_tau = {}
for tau_m in tau_mins_grid:
    tau_d = tau_m / 1440.0
    n_span = max(3, int(np.ceil(4.0 * tau_m / 2.0)))
    trans_ratios = []
    for idx0 in RNG.integers(200, n_points - n_span - 200, size=40):
        win_slice = slice(idx0, idx0 + n_span)
        t_win = time[win_slice]
        t_flare_start = t_win[0]
        t_a = np.maximum(t_win - 0.5 * dt_cad_days, t_flare_start)
        t_b = t_win + 0.5 * dt_cad_days
        flare_binned = (1.5 * max_model_flux) * (tau_d / dt_cad_days) * (
            np.exp(-(t_a - t_flare_start) / tau_d) - np.exp(-(t_b - t_flare_start) / tau_d)
        )
        flux_test = flux.copy()
        flux_test[win_slice] += flare_binned
        fl_s = flux_test[sort_idx_orb]
        fl_pad = np.concatenate([fl_s[-n_pad:], fl_s, fl_s[:n_pad]])
        mod_test = savgol_filter(fl_pad, window_length=w_len_4pct, polyorder=3)[n_pad:n_pad + n_points][unsort_idx_orb]
        res_test = flux_test - mod_test
        recovered_flare_peak = (res_test[idx0] - residuals[idx0]) / flare_binned[0]
        trans_ratios.append(recovered_flare_peak)
    sg_transmission_by_tau[tau_m] = float(np.mean(trans_ratios))

recovery_matrix_single = np.zeros((len(tau_mins_grid), len(amp_frac_grid)))
recovery_matrix_matched = np.zeros((len(tau_mins_grid), len(amp_frac_grid)))

for i_t, tau_m in enumerate(tau_mins_grid):
    tau_d = tau_m / 1440.0
    n_span = max(3, int(np.ceil(4.0 * tau_m / 2.0)))
    t_sg = sg_transmission_by_tau[tau_m]

    for j_a, a_frac in enumerate(amp_frac_grid):
        amp_norm = a_frac * max_model_flux
        det_single = 0
        det_matched = 0

        inj_indices = RNG.integers(100, n_points - n_span - 100, size=n_inj_trials)
        sub_phases = RNG.uniform(0.0, dt_cad_days, size=n_inj_trials)

        for idx0, dt_sub in zip(inj_indices, sub_phases):
            win_slice = slice(idx0, idx0 + n_span)
            t_win = time[win_slice]
            t_flare_start = t_win[0] - 0.5 * dt_cad_days + dt_sub

            t_bin_start = t_win - 0.5 * dt_cad_days
            t_bin_end = t_win + 0.5 * dt_cad_days

            flare_binned = np.zeros_like(t_win)
            active = t_bin_end > t_flare_start
            if np.any(active):
                t_a = np.maximum(t_bin_start[active], t_flare_start)
                t_b = t_bin_end[active]
                flare_binned[active] = t_sg * amp_norm * (tau_d / dt_cad_days) * (
                    np.exp(-(t_a - t_flare_start) / tau_d) - np.exp(-(t_b - t_flare_start) / tau_d)
                )

            res_orig = residuals[win_slice]
            res_inj = res_orig + flare_binned
            newly_triggered = (res_inj > (med_res + 3.0 * std_res)) & (res_orig <= (med_res + 3.0 * std_res))
            if np.any(newly_triggered):
                det_single += 1

            err_win = sigma_rescaled[win_slice]
            if np.sum(flare_binned) > 0:
                template = flare_binned / np.max(flare_binned)
                snr_mf = np.sum(res_inj * template / err_win**2) / np.sqrt(np.sum(template**2 / err_win**2))
                snr_orig = np.sum(res_orig * template / err_win**2) / np.sqrt(np.sum(template**2 / err_win**2))
                if (snr_mf > MF_THRESHOLD_CALIB) and (snr_orig <= MF_THRESHOLD_CALIB):
                    det_matched += 1

        recovery_matrix_single[i_t, j_a] = det_single / n_inj_trials
        recovery_matrix_matched[i_t, j_a] = det_matched / n_inj_trials

eff_exp_90 = 0.90 * total_exposure_days
r90_k0 = 2.302585 / eff_exp_90
r90_k1 = 3.889720 / eff_exp_90

print(f"      [✓] Completeness-Weighted Exposure at A_90: 0.90 x {total_exposure_days:.2f} d = {eff_exp_90:.2f} d -> R90(k=0) < {r90_k0:.4f}/d, R90(k=1) < {r90_k1:.4f}/d")
for i_t, tau_m in enumerate(tau_mins_grid):
    eff_s = recovery_matrix_single[i_t, :]
    eff_m = recovery_matrix_matched[i_t, :]
    eff_at_0345 = eff_s[1] * 100.0
    a50_s = float(np.interp(0.50, eff_s, amp_frac_grid)) if np.max(eff_s) >= 0.50 else 3.50
    a90_s = float(np.interp(0.90, eff_s, amp_frac_grid)) if np.max(eff_s) >= 0.90 else 3.50
    a90_m = float(np.interp(0.90, eff_m, amp_frac_grid)) if np.max(eff_m) >= 0.90 else 3.50
    print(f"      τ = {tau_m:4.0f}m | T_SG = {sg_transmission_by_tau[tau_m]*100:.2f}% | Eff(0.345) = {eff_at_0345:5.1f}% | "
          f"A_50 = {a50_s:.2f} F_pk | A_90(single) = {a90_s:.2f} F_pk ({a90_s*f_peak_phys_es:5.1f} e-/s) | "
          f"A_90(MF) = {a90_m:.2f} F_pk ({a90_m*f_peak_phys_es:5.1f} e-/s)")

# ==============================================================================
# STEP 8: PER-SECTOR & SAME-CCD CONTROL STAR RED NOISE + PIXEL-LEVEL EVENT CHECK
# ==============================================================================
print("\n[8/9] Evaluating Per-Sector Red Noise, Control Aperture & Pixel-Level Event...")

freq_res, power_res = LombScargle(time, residuals).autopower(minimum_frequency=0.5, maximum_frequency=300.0)
log_f_all, log_p_all = np.log10(freq_res), np.log10(power_res)
hf_mask = (freq_res >= 10.0) & (freq_res <= 300.0)
popt_unbin, pcov_unbin = curve_fit(lambda f, a, c: -a * f + c, log_f_all[hf_mask], log_p_all[hf_mask])
alpha_unbin, sigma_alpha_unbin = popt_unbin[0], np.sqrt(np.diag(pcov_unbin))[0]

def compute_log_binned_psd_slope(t_in, r_in):
    f_in, p_in = LombScargle(t_in, r_in).autopower(minimum_frequency=0.5, maximum_frequency=300.0)
    lf_in, lp_in = np.log10(f_in), np.log10(p_in)
    edges = np.linspace(np.log10(0.5), np.log10(300.0), 29)
    bf, bp, berr = [], [], []
    for k in range(28):
        m = (lf_in >= edges[k]) & (lf_in < edges[k + 1])
        if np.sum(m) >= 5:
            bf.append(np.mean(lf_in[m]))
            mp = np.mean(p_in[m])
            sp = np.std(p_in[m]) / np.sqrt(np.sum(m))
            bp.append(np.log10(mp))
            berr.append(sp / (mp * np.log(10.0)))
    bf, bp, berr = np.array(bf), np.array(bp), np.array(berr)
    m_lf = (10**bf >= 0.5) & (10**bf <= 25.0)
    popt, pcov = curve_fit(lambda f, a, c: -a * f + c, bf[m_lf], bp[m_lf], sigma=berr[m_lf], absolute_sigma=False)
    return bf, bp, berr, popt, float(np.sqrt(np.diag(pcov))[0])

bin_logf, bin_logp, bin_logp_err, popt_lf, sigma_alpha_lf = compute_log_binned_psd_slope(time, residuals)
alpha_lf = popt_lf[0]
_, _, _, popt_lf_trim, sigma_alpha_lf_trim = compute_log_binned_psd_slope(time[clean_orbit_mask], residuals[clean_orbit_mask])
alpha_lf_trim = popt_lf_trim[0]

bin_durations_min = [6.0, 10.0, 20.0, 30.0, 60.0, 120.0]

def compute_single_sector_binned_rms(t_s, r_s, dt_min):
    dt_days = dt_min / 1440.0
    n_min_pts = max(2, int(np.round(dt_min / 2.0 * 0.6)))
    edges = np.arange(t_s[0], t_s[-1] + dt_days, dt_days)
    idx_bin = np.digitize(t_s, edges)
    counts = np.bincount(idx_bin, minlength=len(edges) + 1)
    sums = np.bincount(idx_bin, weights=r_s, minlength=len(edges) + 1)
    valid_b = counts >= n_min_pts
    return sums[valid_b] / counts[valid_b]

n_perm = 400
t_cl = time[clean_orbit_mask]
r_cl = residuals[clean_orbit_mask]
sec_cl = sector_id[clean_orbit_mask]

binned_rms_full, binned_rms_clean, white_mean_clean, white_std_clean = [], [], [], []
binned_s70_clean, binned_s92_clean = [], []
z_s70_clean, z_s92_clean, z_comp70_clean, z_comp92_clean = [], [], [], []

for dt_m in bin_durations_min:
    bm_f70 = compute_single_sector_binned_rms(time[sector_id == 70], residuals[sector_id == 70], dt_m)
    bm_f92 = compute_single_sector_binned_rms(time[sector_id == 92], residuals[sector_id == 92], dt_m)
    rms_obs_f = float(np.std(np.concatenate([bm_f70, bm_f92])))

    t70_c, r70_c = t_cl[sec_cl == 70], r_cl[sec_cl == 70]
    t92_c, r92_c = t_cl[sec_cl == 92], r_cl[sec_cl == 92]
    bm_c70 = compute_single_sector_binned_rms(t70_c, r70_c, dt_m)
    bm_c92 = compute_single_sector_binned_rms(t92_c, r92_c, dt_m)

    rms_c70, rms_c92 = float(np.std(bm_c70)), float(np.std(bm_c92))
    rms_obs_c = float(np.std(np.concatenate([bm_c70, bm_c92])))

    tc70 = control_residuals_by_sec[70]['time'][control_residuals_by_sec[70]['clean_mask']]
    rc70 = control_residuals_by_sec[70]['res_comp'][control_residuals_by_sec[70]['clean_mask']]
    tc92 = control_residuals_by_sec[92]['time'][control_residuals_by_sec[92]['clean_mask']]
    rc92 = control_residuals_by_sec[92]['res_comp'][control_residuals_by_sec[92]['clean_mask']]
    rms_comp70 = float(np.std(compute_single_sector_binned_rms(tc70, rc70, dt_m)))
    rms_comp92 = float(np.std(compute_single_sector_binned_rms(tc92, rc92, dt_m)))

    perm_c, perm_70, perm_92, perm_comp70, perm_comp92 = [], [], [], [], []
    for _ in range(n_perm):
        p_c70 = compute_single_sector_binned_rms(t70_c, RNG.permutation(r70_c), dt_m)
        p_c92 = compute_single_sector_binned_rms(t92_c, RNG.permutation(r92_c), dt_m)
        perm_70.append(float(np.std(p_c70)))
        perm_92.append(float(np.std(p_c92)))
        perm_c.append(float(np.std(np.concatenate([p_c70, p_c92]))))
        perm_comp70.append(float(np.std(compute_single_sector_binned_rms(tc70, RNG.permutation(rc70), dt_m))))
        perm_comp92.append(float(np.std(compute_single_sector_binned_rms(tc92, RNG.permutation(rc92), dt_m))))

    z_c = (rms_obs_c - np.mean(perm_c)) / max(np.std(perm_c), 1e-6)
    z_70 = (rms_c70 - np.mean(perm_70)) / max(np.std(perm_70), 1e-6)
    z_92 = (rms_c92 - np.mean(perm_92)) / max(np.std(perm_92), 1e-6)
    zc_70 = (rms_comp70 - np.mean(perm_comp70)) / max(np.std(perm_comp70), 1e-6)
    zc_92 = (rms_comp92 - np.mean(perm_comp92)) / max(np.std(perm_comp92), 1e-6)

    binned_rms_full.append(rms_obs_f)
    binned_rms_clean.append(rms_obs_c)
    white_mean_clean.append(float(np.mean(perm_c)))
    white_std_clean.append(float(np.std(perm_c)))
    binned_s70_clean.append(rms_c70)
    binned_s92_clean.append(rms_c92)
    z_s70_clean.append(z_70)
    z_s92_clean.append(z_92)
    z_comp70_clean.append(zc_70)
    z_comp92_clean.append(zc_92)

    print(f"      Bin {dt_m:4.0f}m: Target S70 = {z_70:+5.1f}σ (Control S70 = {zc_70:+5.1f}σ) | "
          f"Target S92 = {z_92:+5.1f}σ (Control S92 = {zc_92:+5.1f}σ) | Target Pooled = {z_c:+5.1f}σ")

evt_mask = (time >= 3209.98) & (time <= 3210.05)
base_mask = ((time >= 3209.70) & (time < 3209.95)) | ((time > 3210.08) & (time <= 3210.35))
ph_evt_start = ((3209.98 - TASC_BTJD) / detected_period_days) % 1.0
ph_evt_end = ((3210.05 - TASC_BTJD) / detected_period_days) % 1.0
evt_sap_direct_es = float(np.mean(residuals_sap_phys[evt_mask]))
s70_sap_res = residuals_sap_phys[sector_id == 70]
n_blocks_70 = len(s70_sap_res) // 50
evt_sap_red_err = float(np.std(np.mean(s70_sap_res[:n_blocks_70 * 50].reshape(n_blocks_70, 50), axis=1)))

if 70 in tpf_results:
    t_tpf70 = tpf_results[70]['tpf_time']
    fc70 = tpf_results[70]['flux_cube']
    ec70 = tpf_results[70]['err_cube']
    ap70 = tpf_results[70]['aperture_mask']
    m_tpf_evt = (t_tpf70 >= 3209.98) & (t_tpf70 <= 3210.05)
    ph_tpf70 = ((t_tpf70 - TASC_BTJD) / detected_period_days) % 1.0
    m_tpf_ph_base = (t_tpf70 >= 3208.8) & (t_tpf70 <= 3211.2) & (~m_tpf_evt) & (ph_tpf70 >= ph_evt_start) & (ph_tpf70 <= ph_evt_end)
    evt_diff_img = np.nanmean(fc70[m_tpf_evt], axis=0) - np.nanmean(fc70[m_tpf_ph_base], axis=0)
    evt_diff_err = np.sqrt(
        np.nansum(ec70[m_tpf_evt]**2, axis=0) / np.sum(m_tpf_evt)**2 +
        np.nansum(ec70[m_tpf_ph_base]**2, axis=0) / np.sum(m_tpf_ph_base)**2
    )
    evt_ap_flux_es = float(np.nansum(evt_diff_img[ap70]))
    evt_ap_snr = float(evt_ap_flux_es / np.sqrt(np.nansum(evt_diff_err[ap70]**2)))
    ny, nx = evt_diff_img.shape
    y_g, x_g = np.mgrid[0:ny, 0:nx]
    tpx, tpy = tpf_results[70]['target_px'], tpf_results[70]['target_py']
    b_evt = (np.abs(x_g - np.round(tpx)) <= 1) & (np.abs(y_g - np.round(tpy)) <= 1)
    w_evt = np.maximum(evt_diff_img[b_evt], 0.0)
    cx_evt = float(np.sum(x_g[b_evt] * w_evt) / np.sum(w_evt))
    cy_evt = float(np.sum(y_g[b_evt] * w_evt) / np.sum(w_evt))
    evt_offset_arcsec = float(np.hypot(cx_evt - tpx, cy_evt - tpy) * 21.0)
    print(f"      [✓] Pixel-Level TPF Inspection of BTJD 3209.98-3210.05 Event (Phase-Matched Diff Image):")
    print(f"          TPF Aperture Excess = +{evt_ap_flux_es:.2f} e-/s (Photon S/N = {evt_ap_snr:.1f}, Red-Noise S/N = {evt_sap_direct_es/evt_sap_red_err:.2f}σ) | Centroid Offset = {evt_offset_arcsec:.2f}\"")

# ==============================================================================
# STEP 9: RENDER & SAVE ALL 4 PUBLICATION FIGURES (300 DPI)
# ==============================================================================
print("\n[9/9] Rendering Publication Figures (Figures 1 to 4, 300 DPI)...")

fig1 = plt.figure(figsize=(10.8, 11.8), dpi=300)
gs1 = fig1.add_gridspec(3, 2, height_ratios=[1.22, 1.02, 0.92], hspace=0.30, wspace=0.22)

ax0 = fig1.add_subplot(gs1[0, :])
m70, m92 = (sector_id == 70), (sector_id == 92)
ax0.scatter(phase_orb[m70], flux[m70], color='#1f77b4', alpha=0.18, s=2, rasterized=True, label='Sector 70 (2023, CROWDSAP=0.30)')
ax0.scatter(phase_orb[m92], flux[m92], color='#9467bd', alpha=0.14, s=2, rasterized=True, label='Sector 92 (2025, CROWDSAP=0.13)')
ax0.plot(phase_orb_sorted, smooth_heating_sorted, color='#d62728', lw=2.4, label='Wrapped Savitzky-Golay Model (4%)')
s_f_idx = np.argsort(phase_orb)
ax0.plot(phase_orb[s_f_idx], model_fourier[s_f_idx], color='black', ls='--', lw=1.5, label='4-Harmonic Fourier Series')
ax0.axvline(0.25, color='navy', ls=':', alpha=0.6, label=r'Inferior Conj. ($\phi_{\mathrm{orb}}=0.25$, Night-Side)')
ax0.axvline(0.75, color='darkred', ls=':', alpha=0.6, label=r'Superior Conj. ($\phi_{\mathrm{orb}}=0.75$, Day-Side)')
ax0.set_ylabel(r'Normalized Flux ($F / \langle F \rangle$)', fontweight='bold')
ax0.set_title(rf'(a) Ephemeris-Folded TESS Light Curve of PSR J2339-0533 ($P_{{\mathrm{{orb}}}} = {detected_period_days:.8f}({err_digits_period})$ d)', fontweight='bold', loc='left')
ax0.set_xlim(0, 1)
ax0.legend(loc='upper left', ncol=2, fontsize=8.8, frameon=True)
ax0.grid(True, linestyle=':', alpha=0.5)

ax1 = fig1.add_subplot(gs1[1, :], sharex=ax0)
ax1.scatter(phase_orb, pulls, color='gray', alpha=0.18, s=2, rasterized=True, label=r'Standardized Pulls ($z_i = \Delta F_i / \sigma_i$)')
ax1.scatter(phase_orb[pos_3s_pull], pulls[pos_3s_pull], color='#ff7f0e', s=14, alpha=0.9, label=rf'$z > +3\sigma$ ($N={len(pos_3s_pull)}$)')
ax1.scatter(phase_orb[neg_3s_pull], pulls[neg_3s_pull], color='#1f77b4', s=14, alpha=0.9, label=rf'$z < -3\sigma$ ($N={len(neg_3s_pull)}$)')
ax1.axhline(3.0, color='#ff7f0e', ls='--', alpha=0.8)
ax1.axhline(-3.0, color='#1f77b4', ls='--', alpha=0.8)
ax1.set_ylabel(r'Standardized Pull ($z = \Delta F / \sigma$)', fontweight='bold')
ax1.set_title(rf'(b) Error-Normalized Standardized Pulls Across Orbital Phase (Kuiper $p = {kuiper_p:.2f}$)', fontweight='bold', loc='left')
ax1.legend(loc='upper right', ncol=3, fontsize=8.8, frameon=True)
ax1.grid(True, linestyle=':', alpha=0.5)

ax2 = fig1.add_subplot(gs1[2, 0])
bins_z = np.linspace(-5.0, 5.0, 45)
ax2.hist(pulls, bins=bins_z, density=True, color='slategray', alpha=0.65, edgecolor='black', label=r'Observed Pulls ($z_i$)')
x_gauss = np.linspace(-5.0, 5.0, 400)
ax2.plot(x_gauss, norm.pdf(x_gauss, 0, 1), color='#d62728', lw=2.0, label=r'Unit Gaussian $\mathcal{N}(0,1)$')
n_pos_pull = len(pos_3s_pull)
n_neg_pull = len(neg_3s_pull)
ax2.axvline(3.0, color='#ff7f0e', ls='--', lw=1.4, label=rf'$+3\sigma$: {n_pos_pull} (Exp 39.8)')
ax2.axvline(-3.0, color='#1f77b4', ls='--', lw=1.4, label=rf'$-3\sigma$: {n_neg_pull} (Exp 39.8)')
ax2.set_yscale('log')
ax2.set_ylim(1e-4, 1.0)
ax2.set_xlabel(r'Standardized Pull ($z_i = \Delta F_i / \sigma_i$)', fontweight='bold')
ax2.set_ylabel('Probability Density (log)', fontweight='bold')
ax2.set_title(r'(c) Standardized Pull Tails ($+3\sigma=41, -3\sigma=46$)', fontweight='bold', loc='left', fontsize=9.8)
ax2.legend(loc='upper right', fontsize=7.6, frameon=True)
ax2.grid(True, linestyle=':', alpha=0.5)

ax3 = fig1.add_subplot(gs1[2, 1])
bins_mc = np.arange(60, 125, 2)
n_pos_glob = len(pos_3s_global)
n_neg_glob = len(neg_3s_global)
ax3.hist(mc_pos3, bins=bins_mc, density=True, color='#ff7f0e', alpha=0.45, edgecolor='#d62728', label=r'MC $+3\sigma_{\mathrm{res}}$ ($90$ [76-106])')
ax3.hist(mc_neg3, bins=bins_mc, density=True, color='#1f77b4', alpha=0.35, edgecolor='navy', label=r'MC $-3\sigma_{\mathrm{res}}$ ($90$ [76-105])')
ax3.axvline(n_pos_glob, color='#d62728', lw=2.2, ls='-', label=rf'Obs $+3\sigma_{{\mathrm{{res}}}} = {n_pos_glob}$ ($p={p_mc_pos3:.2f}$)')
ax3.axvline(n_neg_glob, color='navy', lw=2.2, ls='--', label=rf'Obs $-3\sigma_{{\mathrm{{res}}}} = {n_neg_glob}$ ($p={p_mc_neg3:.2f}$)')
ax3.set_xlabel(r'Global Unscaled $3\sigma_{\mathrm{res}}$ Outlier Count per Trial', fontweight='bold')
ax3.set_ylabel('MC Probability Density', fontweight='bold')
ax3.set_title(r'(d) Global $\Delta F$ Outliers vs. 3000-Trial MC', fontweight='bold', loc='left', fontsize=9.8)
ax3.legend(loc='upper right', fontsize=7.6, frameon=True)
ax3.grid(True, linestyle=':', alpha=0.5)

plt.savefig('J2339_LightCurve_Residuals.png', bbox_inches='tight', dpi=300)
plt.close(fig1)
print("      [✓] Saved: J2339_LightCurve_Residuals.png (With separated Pull & Global MC panels!)")


fig2, axes2 = plt.subplots(1, 3, figsize=(13.5, 4.3), dpi=300)

for idx_s, sec in enumerate([70, 92]):
    ax_p = axes2[idx_s]
    if sec in tpf_results:
        tr = tpf_results[sec]
        offset_val = tr['offset_arcsec']
        tot_err_val = tr['tot_err_arcsec']
        diff_snr_val = tr['diff_snr_ap']
        crowd_val = sector_data[sec]['crowdsap']
        panel_lbl = 'a' if idx_s == 0 else 'b'

        im = ax_p.imshow(tr['snr_img'], origin='lower', cmap='viridis', aspect='equal')
        plt.colorbar(im, ax=ax_p, fraction=0.046, pad=0.04, label=r'Difference S/N ($\Delta F / \sigma_{\Delta F}$)')
        ax_p.plot(tr['target_px'], tr['target_py'], marker='*', color='red', markersize=14, markeredgecolor='white', label='PSR J2339-0533 (Gaia)')
        ax_p.plot(tr['centroid_px'], tr['centroid_py'], marker='+', color='cyan', markersize=12, mew=2.2,
                  label=rf'Centroid ($\Delta={offset_val:.1f}^{{\prime\prime}}\pm{tot_err_val:.1f}^{{\prime\prime}}$)')
        ax_p.set_title(f'({panel_lbl}) Sector {sec} Max-Min Diff Image\n(CROWDSAP={crowd_val:.2f}, Ap. S/N={diff_snr_val:.0f})', fontweight='bold', fontsize=10)
        ax_p.set_xlabel('Column Pixel', fontweight='bold')
        ax_p.set_ylabel('Row Pixel', fontweight='bold')
        ax_p.legend(loc='lower right', fontsize=7.8, framealpha=0.85)

ax_c = axes2[2]
ph_grid = np.linspace(0, 1, 25)
ph_mid = 0.5 * (ph_grid[:-1] + ph_grid[1:])
for sec, col, mk in [(70, '#1f77b4', 'o'), (92, '#d62728', 's')]:
    m_s = (sector_id == sec)
    b_med = [np.median(flux_phys[m_s][(phase_orb[m_s] >= ph_grid[k]) & (phase_orb[m_s] < ph_grid[k+1])]) for k in range(len(ph_grid)-1)]
    b_err = [np.std(flux_phys[m_s][(phase_orb[m_s] >= ph_grid[k]) & (phase_orb[m_s] < ph_grid[k+1])]) /
             np.sqrt(np.sum((phase_orb[m_s] >= ph_grid[k]) & (phase_orb[m_s] < ph_grid[k+1]))) for k in range(len(ph_grid)-1)]
    pp_val = sector_data[sec]['pp_pdc_es']
    ax_c.errorbar(ph_mid, b_med, yerr=b_err, fmt=mk+'-', color=col, lw=1.8, ms=5, capsize=2,
                  label=rf'S{sec} PDCSAP ($\Delta F_{{\mathrm{{pp}}}}={pp_val:.1f}\ \mathrm{{e^- s^{{-1}}}}$)')

ax_c.set_xlabel(r'Binary Orbital Phase ($\phi_{\mathrm{orb}}$, $T_{\mathrm{asc}}$ Ref.)', fontweight='bold')
ax_c.set_ylabel(r'Physical PDCSAP Flux ($\mathrm{e^- s^{-1}}$)', fontweight='bold')
ax_c.set_title('(c) Sector-by-Sector Physical Flux\n& Peak-to-Peak Amplitude', fontweight='bold', fontsize=10)
ax_c.grid(True, linestyle=':', alpha=0.5)
ax_c.legend(loc='upper left', fontsize=8.3, frameon=True)

plt.tight_layout()
plt.savefig('J2339_Crowding_DifferenceImaging.png', bbox_inches='tight', dpi=300)
plt.close(fig2)
print("      [✓] Saved: J2339_Crowding_DifferenceImaging.png")

# FIGURE 3
fig3, axes3 = plt.subplots(1, 2, figsize=(12.2, 4.8), dpi=300)

colors_tau = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2']
for i_t, tau_m in enumerate(tau_mins_grid):
    axes3[0].plot(amp_frac_grid, recovery_matrix_single[i_t, :] * 100.0, marker='o', lw=2.0, color=colors_tau[i_t],
                  label=rf'$\tau = {tau_m:.0f}$ min')

axes3[0].axvline(0.345, color='black', ls='--', lw=1.4, label=r'Old $1\sigma_{\mathrm{res}}/F_{\mathrm{peak}}$ (0.345)')
axes3[0].axhline(90.0, color='gray', ls=':', lw=1.4, label='90% Recovery Threshold')
axes3[0].set_xlabel(r'Injected Flare Peak Amplitude ($A_{\mathrm{peak}} / F_{\mathrm{peak}}$)', fontweight='bold')
axes3[0].set_ylabel('Detection Efficiency (%)', fontweight='bold')
axes3[0].set_title('(a) Full-Pipeline FRED Flare Recovery Curves', fontweight='bold', loc='left')
axes3[0].set_ylim(-3, 105)
axes3[0].grid(True, linestyle=':', alpha=0.5)
axes3[0].legend(loc='lower right', fontsize=8.5, ncol=2, frameon=True)

amp_edges = np.zeros(len(amp_frac_grid) + 1)
amp_edges[1:-1] = 0.5 * (amp_frac_grid[:-1] + amp_frac_grid[1:])
amp_edges[0] = amp_frac_grid[0] - 0.5 * (amp_frac_grid[1] - amp_frac_grid[0])
amp_edges[-1] = amp_frac_grid[-1] + 0.5 * (amp_frac_grid[-1] - amp_frac_grid[-2])
tau_edges = np.arange(len(tau_mins_grid) + 1) - 0.5

im3 = axes3[1].pcolormesh(amp_edges, tau_edges, recovery_matrix_matched * 100.0, cmap='magma', vmin=0, vmax=100, shading='flat')
plt.colorbar(im3, ax=axes3[1], label=rf'Matched-Filter Efficiency (%, $\mathrm{{S/N}}>{MF_THRESHOLD_CALIB:.2f}$)')
axes3[1].set_yticks(range(len(tau_mins_grid)))
axes3[1].set_yticklabels([f'{int(t)} min' for t in tau_mins_grid])
axes3[1].set_xlabel(r'Injected Flare Peak Amplitude ($A_{\mathrm{peak}} / F_{\mathrm{peak}}$)', fontweight='bold')
axes3[1].set_ylabel(r'Flare $e$-Folding Duration ($\tau$)', fontweight='bold')
axes3[1].set_title('(b) Noise-Calibrated Matched-Filter Completeness', fontweight='bold', loc='left')

plt.tight_layout()
plt.savefig('J2339_InjectionRecovery.png', bbox_inches='tight', dpi=300)
plt.close(fig3)
print("      [✓] Saved: J2339_InjectionRecovery.png")

# FIGURE 4
fig4, axes4 = plt.subplots(1, 3, figsize=(14.2, 4.5), dpi=300)

axes4[0].plot(bin_durations_min, binned_rms_full, 'o--', color='#d62728', lw=1.6, alpha=0.8, label='Pooled RMS (All Cadences)')
axes4[0].plot(bin_durations_min, binned_rms_clean, 's-', color='#2ca02c', lw=2.2, label='Pooled RMS (Trimmed)')
axes4[0].plot(bin_durations_min, binned_s70_clean, '^-', color='#1f77b4', lw=1.8, label='Sector 70 RMS (Trimmed)')
axes4[0].plot(bin_durations_min, binned_s92_clean, 'v-', color='#9467bd', lw=1.8, label='Sector 92 RMS (Trimmed)')
axes4[0].plot(bin_durations_min, white_mean_clean, 'k:', lw=1.8, label='Pooled White-Noise Null')
axes4[0].fill_between(bin_durations_min,
                      np.array(white_mean_clean) - 3*np.array(white_std_clean),
                      np.array(white_mean_clean) + 3*np.array(white_std_clean),
                      color='gray', alpha=0.25, label=r'White-Noise $\pm 3\sigma$ Band')
axes4[0].set_xscale('log')
axes4[0].set_yscale('log')
axes4[0].set_xlabel('Time Bin Size (minutes)', fontweight='bold')
axes4[0].set_ylabel(r'Binned Residual RMS ($\sigma_{\mathrm{bin}}$)', fontweight='bold')
axes4[0].set_title('(a) Per-Sector & Pooled Binned RMS Excess', fontweight='bold', fontsize=10)
axes4[0].grid(True, which='both', ls=':', alpha=0.5)
axes4[0].legend(loc='upper right', fontsize=7.5, frameon=True)

axes4[1].plot(10**log_f_all[::10], 10**log_p_all[::10], '.', color='lightgray', ms=2, alpha=0.5, rasterized=True, label='Raw Unbinned PSD')
axes4[1].errorbar(10**bin_logf, 10**bin_logp, yerr=(10**bin_logp)*bin_logp_err*np.log(10),
                  fmt='o', color='#1f77b4', ms=5, capsize=2, label='Log-Binned PSD (Full)')
f_plot_lf = np.logspace(np.log10(0.5), np.log10(25.0), 100)
axes4[1].plot(f_plot_lf, 10**(popt_lf[1] - alpha_lf * np.log10(f_plot_lf)), color='#d62728', lw=2.2,
              label=rf'Low-Freq Fit ($\alpha = {alpha_lf:.2f} \pm {sigma_alpha_lf:.2f}$)')
f_plot_hf = np.logspace(np.log10(10.0), np.log10(300.0), 100)
axes4[1].plot(f_plot_hf, 10**(popt_unbin[1] - alpha_unbin * np.log10(f_plot_hf)), color='black', ls='--', lw=1.8,
              label=rf'High-Freq Floor ($\alpha = {alpha_unbin:.3f} \pm {sigma_alpha_unbin:.3f}$)')
axes4[1].set_xscale('log')
axes4[1].set_yscale('log')
axes4[1].set_xlabel('Frequency (cycles / day)', fontweight='bold')
axes4[1].set_ylabel('Lomb-Scargle Power', fontweight='bold')
axes4[1].set_title('(b) Log-Binned Power Spectral Density', fontweight='bold', fontsize=10)
axes4[1].grid(True, which='both', ls=':', alpha=0.5)
axes4[1].legend(loc='upper right', fontsize=8, frameon=True)

win_evt = (time >= 3209.65) & (time <= 3210.35)
t_evt = time[win_evt]
f_evt_sap = sap_phys[win_evt]
mod_evt_sap = sap_phys[win_evt] - residuals_sap_phys[win_evt]
f_smooth_15m = uniform_filter1d(f_evt_sap, size=8)

axes4[2].errorbar(t_evt - 3209.0, f_evt_sap, yerr=sap_err_phys[win_evt], fmt='.', color='gray', alpha=0.35, ms=3, label='120-s Direct SAP Flux')
axes4[2].plot(t_evt - 3209.0, f_smooth_15m, color='#1f77b4', lw=2.0, label='16-min Boxcar Mean')
axes4[2].plot(t_evt - 3209.0, mod_evt_sap, color='#d62728', lw=2.0, ls='--', label='Mean SAP Heating Model')
axes4[2].axvspan(3209.98 - 3209.0, 3210.05 - 3209.0, color='gold', alpha=0.35, label=rf'BTJD 3210 Event ($\phi={ph_evt_start:.2f}$-${ph_evt_end:.2f}$)')
axes4[2].set_xlabel('Time (BTJD - 3209.0 days)', fontweight='bold')
axes4[2].set_ylabel(r'Direct Aperture SAP Flux ($\mathrm{e^- s^{-1}}$)', fontweight='bold')
axes4[2].set_title(rf'(c) S70 Event ($\Delta F_{{\mathrm{{SAP}}}}=+{evt_sap_direct_es:.2f} \pm {evt_sap_red_err:.2f}\ \mathrm{{e^- s^{{-1}}}}$)', fontweight='bold', fontsize=9.5)
axes4[2].grid(True, ls=':', alpha=0.5)
axes4[2].legend(loc='upper right', fontsize=7.8, frameon=True)

plt.tight_layout()
plt.savefig('J2339_RedNoise_CandidateEvent.png', bbox_inches='tight', dpi=300)
plt.close(fig4)
print("      [✓] Saved: J2339_RedNoise_CandidateEvent.png")

print("\n==========================================================================")
print("   v6.0 PIPELINE EXECUTION COMPLETE                                       ")
print("==========================================================================")

   TESS OPTICAL VARIABILITY PIPELINE: PSR J2339-0533 (v6.0)         
   Full End-to-End Pipeline with Separated Pull/Global MC & Flux Calib    

[1/9] Querying NASA MAST TESS Archive for PSR J2339-0533 (TIC 2052434280)...

      --- SECTOR-BY-SECTOR METADATA, QUALITY FLAGS & CROWDING SUMMARY ---
      Sector 70 (TIC 2052434280, Cam 1/CCD 1):
        Raw Cadences     : 17888 total | 3587 flagged (QUALITY>0) | 14301 valid finite (100% QUALITY==0: True)
        BTJD Range       : 3208.57 - 3232.96 (14301 valid 120-s cadences)
        CROWDSAP / FLFRC : 0.3032 / 0.4759 (69.7% aperture contamination)
        Median PDCSAP    : 9.38 e-/s (Inferred Tmag = 18.01 vs Catalog 18.61 [5.40 e-/s])
        Median SAP / BKG : 14.74 e-/s | BKG: 306.9 e-/s
        Median Frac Err  : 0.5291 (52.91%) | Median(F/σ): 1.86 | 1/σ_phot: 1.89
      Sector 92 (TIC 2052434280, Cam 4/CCD 3):
        Raw Cadences     : 18756 total | 3546 flagged (QUALITY>0) | 15210 valid finite (100% QUALITY==0: True)
        BTJD 